# Mechanistic Interpretability Workflow Demo

**Mechanistic interpretability** aims to understand *how* language models compute—which circuits, layers, and heads implement specific behaviors. This notebook walks through a minimal workflow to localize where a model "stores" and "uses" factual knowledge.

## Workflow

1. **Load model** — GPT-2 via TransformerLens (small, well-studied)
2. **Capture activations** — Run with hooks to cache intermediate representations (residual stream, attention, MLP outputs)
3. **Visualize** — Logit lens: when does the correct answer first appear in intermediate layers?
4. **Run experiment** — Activation patching: which layers are *causally* important for the task?

See [docs/METHODS.md](../docs/METHODS.md) for how each technique works and what conclusions are valid.

## 1. Load Model

We use **TransformerLens** to load GPT-2 small. TransformerLens provides a clean interface for running models with hooks, caching activations, and patching—essential for interpretability experiments.

In [ ]:
from mechinterp_lab.models import load_model

model = load_model("gpt2", device="cpu")
print(f"Loaded {model.name}: {model.config.n_layers} layers, {model.config.n_heads} heads")

## 2. Capture Activations

**Hooks** let us intercept and cache activations during a forward pass. We run the model on a prompt and store intermediate tensors (residual stream, attention outputs, MLP outputs) at each layer. This cache is what we'll later probe (logit lens) or patch (activation patching).

In [ ]:
from mechinterp_lab.hooks import capture_activations

prompt = "The capital of France is"
tokens = model.model.to_tokens(prompt)
logits, cache = capture_activations(model, tokens)

print(f"Captured {len(cache.cache)} activation tensors")
for name in list(cache.cache.keys())[:5]:
    print(f"  {name}: {cache.cache[name].shape}")

## 3. Visualize: Logit Lens

**Logit lens:** At each layer, we project the residual stream through the unembedding matrix to get "virtual logits"—what the model would predict if it stopped there. The plot shows the *top predicted token* at each layer. When the correct answer (e.g. " Paris") first appears as the top prediction, that suggests the information has entered the residual stream. **Caveat:** This is correlational, not causal; the unembedding is trained for the final residual, not intermediate ones.

In [ ]:
import matplotlib.pyplot as plt

from mechinterp_lab.probes import logit_lens

logits_per_layer, _ = logit_lens(model, tokens, layers=list(range(model.config.n_layers)))

# Get top-1 predicted token per layer
top_logits = logits_per_layer.max(dim=-1).values.squeeze()  # [n_layers]
top_tokens = logits_per_layer.argmax(dim=-1).squeeze()  # [n_layers]
token_strs = [model.model.to_string(t.item()) for t in top_tokens]

plt.figure(figsize=(10, 4))
plt.bar(range(len(token_strs)), top_logits.detach().numpy(), color="steelblue", alpha=0.7)
plt.xticks(range(len(token_strs)), token_strs, rotation=45, ha="right")
plt.xlabel("Layer")
plt.ylabel("Top predicted token")
plt.title("Logit Lens: predicted token at each layer")
plt.tight_layout()
plt.show()

## 4. Run Experiment: Activation Patching

**Activation patching (causal tracing):** We run on two prompts—*clean* (correct: "France is Paris") and *corrupt* (wrong: "France is London"). For each layer, we patch the *corrupt* run's residual with the *clean* run's residual at that layer, then measure how much the output shifts toward the correct answer.

- **Metric:** Logit difference (correct token − wrong token) at the last position. Higher = more causal importance.
- **Interpretation:** Layers with large bars are *causally* involved in producing the correct answer. Compare with logit lens: patching tells you *where it matters*, not just where it appears.

In [ ]:
import torch

from mechinterp_lab.patching import patch_residual_stream

clean = "The capital of France is Paris"
corrupt = "The capital of France is London"
clean_tokens = model.model.to_tokens(clean)
corrupt_tokens = model.model.to_tokens(corrupt)

# Align lengths
min_len = min(clean_tokens.shape[1], corrupt_tokens.shape[1])
clean_tokens = clean_tokens[:, :min_len]
corrupt_tokens = corrupt_tokens[:, :min_len]

_, clean_cache = model.model.run_with_cache(clean_tokens)

def metric_fn(logits: torch.Tensor) -> torch.Tensor:
    last = logits.shape[1] - 1
    correct = clean_tokens[0, last].item()
    wrong = corrupt_tokens[0, last].item()
    return logits[0, last, correct] - logits[0, last, wrong]

patch_effects = patch_residual_stream(model, corrupt_tokens, clean_cache, metric_fn)

plt.figure(figsize=(10, 4))
plt.bar(range(len(patch_effects)), patch_effects.detach().numpy(), color="coral", alpha=0.8)
plt.xlabel("Layer")
plt.ylabel("Logit diff (correct - wrong)")
plt.title("Activation patching: causal effect of each layer's residual")
plt.tight_layout()
plt.show()

## Summary

| Technique | What it tells you | Causal? |
|-----------|------------------|---------|
| **Logit lens** | When the correct answer first appears in intermediate layers | No (correlational) |
| **Activation patching** | Which layers are causally important for the task | Yes |

**Next steps:** Run the CLI with `mechinterp activate-patching`, `activate-patching-multi-fact`, etc. for more experiments. See [docs/METHODS.md](../docs/METHODS.md) for how each method works and what conclusions are valid vs invalid.